In [1]:
"""
════════════════════════════════════════════
Central configuration, terminal logger, shared type definitions,
and memory-tier enums used across the entire LTM-RAG system.

LTM-RAG Memory Architecture:
┌────────────────────────────────────────────────────────────────┐
│  TIER 1 — Working Memory     (current session, in-RAM buffer)  │
│  TIER 2 — Episodic Memory    (past conversations, SQLite+HNSW) │
│  TIER 3 — Semantic Memory    (distilled facts, ChromaDB)       │
│  TIER 4 — Knowledge Base     (static corpus, ChromaDB)         │
└────────────────────────────────────────────────────────────────┘

Inspired by:
  • MemGPT         (Packer et al., 2023)  https://arxiv.org/abs/2310.08560
  • Generative Agents (Park et al., 2023) https://arxiv.org/abs/2304.03442
  • A-MEM          (Xu et al., 2024)      https://arxiv.org/abs/2312.06681
  • HippoRAG       (Gutierrez et al.,2024)https://arxiv.org/abs/2405.14831
"""

'\n════════════════════════════════════════════\nCentral configuration, terminal logger, shared type definitions,\nand memory-tier enums used across the entire LTM-RAG system.\n\nLTM-RAG Memory Architecture:\n┌────────────────────────────────────────────────────────────────┐\n│  TIER 1 — Working Memory     (current session, in-RAM buffer)  │\n│  TIER 2 — Episodic Memory    (past conversations, SQLite+HNSW) │\n│  TIER 3 — Semantic Memory    (distilled facts, ChromaDB)       │\n│  TIER 4 — Knowledge Base     (static corpus, ChromaDB)         │\n└────────────────────────────────────────────────────────────────┘\n\nInspired by:\n  • MemGPT         (Packer et al., 2023)  https://arxiv.org/abs/2310.08560\n  • Generative Agents (Park et al., 2023) https://arxiv.org/abs/2304.03442\n  • A-MEM          (Xu et al., 2024)      https://arxiv.org/abs/2312.06681\n  • HippoRAG       (Gutierrez et al.,2024)https://arxiv.org/abs/2405.14831\n'

In [2]:

# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────
from __future__ import annotations

import os
import re
import ast
import sys
import json
import math
import time
import hashlib
import textwrap
import warnings
import operator
import datetime
import traceback
from enum import Enum
from pathlib import Path
from typing import (
    Any, Callable, Dict, List, Optional,
    Tuple, Union, NamedTuple
)
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")


In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

W0525 08:30:37.401000 35580 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [6]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_classic.agents import AgentExecutor
from langgraph.checkpoint.memory import InMemorySaver



In [7]:
import chromadb
from chromadb.config import Settings


In [3]:

# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Ollama (local LLM) ─────────────────────────────────────────────
    OLLAMA_BASE_URL: str = field(default_factory=lambda: os.getenv(
        "OLLAMA_BASE_URL", "http://localhost:11434"))
    OLLAMA_MODEL: str    = field(default_factory=lambda: os.getenv(
        "OLLAMA_MODEL", "qwen2.5-coder:7b"))

    # ── HuggingFace Embeddings (local) ───────────────────────────────
    EMBEDDING_MODEL: str   = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str  = field(default_factory=lambda: os.getenv(
        "EMBEDDING_DEVICE", "cpu"))
    EMBEDDING_DIM: int     = 384   # all-MiniLM-L6-v2 output dimension

    # ── Storage Paths ────────────────────────────────────────────────
    BASE_DIR: str                 = "./ltm_rag_store"
    EPISODIC_DB_PATH: str         = "./ltm_rag_store/episodic.db"
    SEMANTIC_CHROMA_DIR: str      = "./ltm_rag_store/semantic_chroma"
    KB_CHROMA_DIR: str            = "./ltm_rag_store/kb_chroma"
    USER_PROFILE_PATH: str        = "./ltm_rag_store/user_profiles.json"

    # ── ChromaDB Collections ─────────────────────────────────────────
    SEMANTIC_COLLECTION: str      = "ltm_semantic_memory"
    KB_COLLECTION: str            = "ltm_knowledge_base"
    EPISODIC_COLLECTION: str      = "ltm_episodic_memory"

    # ── Retrieval ────────────────────────────────────────────────────
    TOP_K_SEMANTIC: int    = 4
    TOP_K_EPISODIC: int    = 3
    TOP_K_KB: int          = 4
    FETCH_K: int           = 10
    MMR_LAMBDA: float      = 0.65
    CHUNK_SIZE: int        = 700
    CHUNK_OVERLAP: int     = 120

    # ── Working Memory ────────────────────────────────────────────────
    WORKING_MEMORY_MAX_TURNS: int = 8    # turns kept in active context
    WORKING_MEMORY_MAX_TOKENS: int = 2000

    # ── Memory Consolidation ──────────────────────────────────────────
    # Episodic episodes are consolidated → semantic facts when accessed ≥ threshold
    CONSOLIDATION_ACCESS_THRESHOLD: int = 2
    # Episodes older than N days decay in relevance
    DECAY_HALFLIFE_DAYS: float    = 30.0
    # Semantic facts reinforced each access
    REINFORCE_DELTA: float        = 0.1
    # Importance score threshold for storing a memory
    MIN_IMPORTANCE_SCORE: float   = 0.3
    # Max semantic memories per user
    MAX_SEMANTIC_PER_USER: int    = 500
    # Max episodic episodes per user
    MAX_EPISODIC_PER_USER: int    = 1000

    # ── LLM ──────────────────────────────────────────────────────────
    LLM_TEMPERATURE: float = 0.0
    LLM_MAX_TOKENS: int    = 1200

    def is_configured(self) -> bool:
        """Check Ollama is reachable."""
        import urllib.request
        try:
            urllib.request.urlopen(self.OLLAMA_BASE_URL, timeout=3)
            return True
        except Exception:
            return False
# ══════════════════════════════════════════════════════════════════════
# MEMORY TIER ENUM
# ══════════════════════════════════════════════════════════════════════

class MemoryTier(str, Enum):
    WORKING   = "working"    # current session RAM buffer
    EPISODIC  = "episodic"   # past conversation episodes
    SEMANTIC  = "semantic"   # distilled facts + preferences
    KB        = "kb"         # static knowledge base documents


class MemoryOperation(str, Enum):
    STORE     = "store"
    RETRIEVE  = "retrieve"
    UPDATE    = "update"
    FORGET    = "forget"
    CONSOLIDATE = "consolidate"
    REINFORCE = "reinforce"


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# TYPED MEMORY RECORDS
# ══════════════════════════════════════════════════════════════════════

@dataclass
class EpisodicRecord:
    """A single conversation episode stored in episodic memory."""
    episode_id:     str
    user_id:        str
    session_id:     str
    role:           str          # "user" | "assistant"
    content:        str
    timestamp:      float        = field(default_factory=time.time)
    importance:     float        = 0.5   # 0–1, higher = more important
    access_count:   int          = 0
    last_accessed:  float        = field(default_factory=time.time)
    consolidated:   bool         = False  # True if promoted to semantic
    embedding:      Optional[List[float]] = None
    metadata:       Dict[str, Any] = field(default_factory=dict)

    def age_days(self) -> float:
        return (time.time() - self.timestamp) / 86400.0

    def decayed_importance(self, halflife_days: float) -> float:
        """Exponential decay: importance × 0.5^(age/halflife)."""
        return self.importance * (0.5 ** (self.age_days() / halflife_days))


In [10]:


@dataclass
class SemanticRecord:
    """A distilled fact or preference in semantic memory."""
    fact_id:        str
    user_id:        str
    content:        str          # the distilled fact or preference
    fact_type:      str          # "preference" | "fact" | "skill" | "goal" | "identity"
    source_episode: Optional[str] = None   # episode_id that originated this
    timestamp:      float        = field(default_factory=time.time)
    strength:       float        = 0.7    # 0–1, reinforced each access
    access_count:   int          = 0
    last_accessed:  float        = field(default_factory=time.time)
    confidence:     float        = 0.8    # how confident we are in this fact
    metadata:       Dict[str, Any] = field(default_factory=dict)

    def reinforce(self, delta: float = 0.1):
        """Strengthen this memory on retrieval."""
        self.strength = min(1.0, self.strength + delta)
        self.access_count += 1
        self.last_accessed = time.time()



In [11]:

@dataclass
class WorkingMemoryTurn:
    """One turn in the current session's working memory."""
    turn_id:   str
    role:      str        # "user" | "assistant"
    content:   str
    timestamp: float = field(default_factory=time.time)
    metadata:  Dict[str, Any] = field(default_factory=dict)


@dataclass
class UserProfile:
    """Persistent user profile updated across sessions."""
    user_id:          str
    display_name:     str          = ""
    created_at:       float        = field(default_factory=time.time)
    last_active:      float        = field(default_factory=time.time)
    total_sessions:   int          = 0
    total_turns:      int          = 0
    preferences:      Dict[str, Any] = field(default_factory=dict)
    # e.g. {"language": "English", "verbosity": "concise", "topics": [...]}
    known_facts:      List[str]    = field(default_factory=list)   # quick-access list
    metadata:         Dict[str, Any] = field(default_factory=dict)

    def update_activity(self):
        self.last_active  = time.time()
        self.total_turns += 1


In [12]:

@dataclass
class MemoryRetrievalResult:
    """Aggregated result from all four memory tiers."""
    query:              str
    user_id:            str
    working_context:    List[WorkingMemoryTurn] = field(default_factory=list)
    episodic_results:   List[EpisodicRecord]    = field(default_factory=list)
    semantic_results:   List[SemanticRecord]    = field(default_factory=list)
    kb_results:         List[Any]               = field(default_factory=list)  # LangChain Documents
    user_profile:       Optional[UserProfile]   = None
    retrieval_time_ms:  int                     = 0
    tiers_used:         List[MemoryTier]        = field(default_factory=list)

    def format_context(self, cfg: Config) -> str:
        """Format all memory tiers into a single LLM-ready context string."""
        parts: List[str] = []

        # User profile summary
        if self.user_profile:
            prefs = self.user_profile.preferences
            if prefs:
                pref_str = ", ".join(f"{k}={v}" for k, v in list(prefs.items())[:5])
                parts.append(f"[USER PROFILE] {self.user_profile.display_name or self.user_id} "
                              f"| Sessions: {self.user_profile.total_sessions} "
                              f"| Preferences: {pref_str}")

        # Semantic memory (high-level facts/preferences)
        if self.semantic_results:
            sem_lines = [f"  • [{r.fact_type.upper()}] {r.content} (strength={r.strength:.2f})"
                         for r in self.semantic_results[:4]]
            parts.append("[LONG-TERM SEMANTIC MEMORY]\n" + "\n".join(sem_lines))

        # Episodic memory (past conversations)
        if self.episodic_results:
            ep_lines = []
            for r in self.episodic_results[:3]:
                age = f"{r.age_days():.0f}d ago"
                ep_lines.append(f"  • [{r.role.upper()} — {age}] {r.content[:200]}")
            parts.append("[EPISODIC MEMORY — Past Conversations]\n" + "\n".join(ep_lines))

        # Knowledge base
        if self.kb_results:
            kb_lines = []
            for i, doc in enumerate(self.kb_results[:4], 1):
                src = doc.metadata.get("source", "Unknown")
                kb_lines.append(f"  [{i}] {src}\n      {doc.page_content.strip()[:300]}")
            parts.append("[KNOWLEDGE BASE]\n" + "\n".join(kb_lines))

        # Working memory (current session)
        if self.working_context:
            wm_lines = [f"  {t.role.upper()}: {t.content[:150]}"
                        for t in self.working_context[-4:]]
            parts.append("[CURRENT SESSION]\n" + "\n".join(wm_lines))

        return "\n\n".join(parts) if parts else "(no memory context available)"



In [13]:

# ══════════════════════════════════════════════════════════════════════
# TERMINAL LOGGER
# ══════════════════════════════════════════════════════════════════════

class C:
    RESET  = "\033[0m"; BOLD   = "\033[1m"; DIM    = "\033[2m"
    CYAN   = "\033[96m"; GREEN = "\033[92m"; YELLOW = "\033[93m"
    RED    = "\033[91m"; MAG   = "\033[95m"; BLUE   = "\033[94m"
    WHITE  = "\033[97m"; ORANGE= "\033[33m"
    # Memory tier colours
    WORK   = "\033[94m"   # blue   — Working memory
    EPIS   = "\033[92m"   # green  — Episodic memory
    SEM    = "\033[95m"   # magenta— Semantic memory
    KB     = "\033[96m"   # cyan   — Knowledge base
    CONS   = "\033[93m"   # yellow — Consolidation
    PROF   = "\033[33m"   # orange — User profile

W = 76


class Log:
    @staticmethod
    def banner(title: str):
        print(f"\n{C.BOLD}{C.WHITE}{'═'*W}")
        print(f"  {title}")
        print(f"{'═'*W}{C.RESET}")

    @staticmethod
    def section(title: str, colour: str = C.CYAN):
        print(f"\n{C.BOLD}{colour}{'━'*W}")
        print(f"  {title}")
        print(f"{'━'*W}{C.RESET}")

    @staticmethod
    def tier(tier: MemoryTier, msg: str):
        colours = {
            MemoryTier.WORKING:  C.WORK,
            MemoryTier.EPISODIC: C.EPIS,
            MemoryTier.SEMANTIC: C.SEM,
            MemoryTier.KB:       C.KB,
        }
        col = colours.get(tier, C.WHITE)
        tag = f"[{tier.value.upper():9s}]"
        print(f"{C.BOLD}{col}{tag}{C.RESET} {msg}")

    @staticmethod
    def step(msg: str):
        print(f"\n{C.BOLD}{C.BLUE}▶ {msg}{C.RESET}")

    @staticmethod
    def ok(msg: str):
        print(f"{C.GREEN}  ✓ {msg}{C.RESET}")

    @staticmethod
    def warn(msg: str):
        print(f"{C.YELLOW}  ⚠  {msg}{C.RESET}")

    @staticmethod
    def err(msg: str):
        print(f"{C.RED}  ✗ {msg}{C.RESET}")

    @staticmethod
    def info(msg: str):
        print(f"{C.DIM}  · {msg}{C.RESET}")

    @staticmethod
    def memory_op(op: MemoryOperation, tier: MemoryTier, detail: str):
        op_icons = {
            MemoryOperation.STORE:       "💾",
            MemoryOperation.RETRIEVE:    "🔍",
            MemoryOperation.UPDATE:      "✏️ ",
            MemoryOperation.FORGET:      "🗑️ ",
            MemoryOperation.CONSOLIDATE: "🔄",
            MemoryOperation.REINFORCE:   "⚡",
        }
        icon = op_icons.get(op, "·")
        colours = {
            MemoryTier.WORKING:  C.WORK,
            MemoryTier.EPISODIC: C.EPIS,
            MemoryTier.SEMANTIC: C.SEM,
            MemoryTier.KB:       C.KB,
        }
        col = colours.get(tier, C.WHITE)
        print(f"{col}{icon} [{op.value:11s}][{tier.value:8s}]{C.RESET} {detail}")

    @staticmethod
    def answer_box(query: str, answer: str, meta: Dict[str, Any]):
        print(f"\n{C.BOLD}{C.GREEN}{'═'*W}")
        print(f"  LTM-RAG ANSWER")
        print(f"{'═'*W}{C.RESET}")
        print(f"{C.BOLD}  Q: {query}{C.RESET}\n")
        import textwrap
        for line in textwrap.wrap(answer, W - 4):
            print(f"  {line}")
        print(f"\n{C.DIM}{'─'*W}")
        print(f"  Memory tiers used  : {[t.value for t in meta.get('tiers_used',[])]}")
        print(f"  Semantic memories  : {meta.get('semantic_count',0)}")
        print(f"  Episodic memories  : {meta.get('episodic_count',0)}")
        print(f"  KB docs retrieved  : {meta.get('kb_count',0)}")
        print(f"  Working mem turns  : {meta.get('working_turns',0)}")
        print(f"  Consolidations     : {meta.get('consolidations',0)}")
        print(f"  Elapsed            : {meta.get('elapsed',0):.2f}s")
        print(f"{'─'*W}{C.RESET}")


In [8]:
"""
════════════════════════════════════════════
Demo corpus: 10 documents covering memory systems, RAG, and cognitive architectures.
All papers are publicly available on arXiv.

📄 Reference PDFs:
  1. MemGPT          https://arxiv.org/pdf/2310.08560  (primary PDF — auto-downloaded)
  2. Generative Agents https://arxiv.org/pdf/2304.03442
  3. A-MEM           https://arxiv.org/pdf/2312.06681
  4. HippoRAG        https://arxiv.org/pdf/2405.14831
  5. MemoryBank       https://arxiv.org/pdf/2305.10250
  6. RAG (Lewis)      https://arxiv.org/pdf/2005.11401
  7. Self-RAG         https://arxiv.org/pdf/2310.11511
  8. Cognitive Architecture (Anderson 1983) — classic reference
  9. LangChain Memory https://python.langchain.com/docs/concepts/memory/
  10. ChromaDB Docs   https://docs.trychroma.com/
"""

'\n════════════════════════════════════════════\nDemo corpus: 10 documents covering memory systems, RAG, and cognitive architectures.\nAll papers are publicly available on arXiv.\n\n📄 Reference PDFs:\n  1. MemGPT          https://arxiv.org/pdf/2310.08560  (primary PDF — auto-downloaded)\n  2. Generative Agents https://arxiv.org/pdf/2304.03442\n  3. A-MEM           https://arxiv.org/pdf/2312.06681\n  4. HippoRAG        https://arxiv.org/pdf/2405.14831\n  5. MemoryBank       https://arxiv.org/pdf/2305.10250\n  6. RAG (Lewis)      https://arxiv.org/pdf/2005.11401\n  7. Self-RAG         https://arxiv.org/pdf/2310.11511\n  8. Cognitive Architecture (Anderson 1983) — classic reference\n  9. LangChain Memory https://python.langchain.com/docs/concepts/memory/\n  10. ChromaDB Docs   https://docs.trychroma.com/\n'

In [9]:

CORPUS: List[Dict[str, str]] = [
    {
        "id": "memgpt",
        "source": "MemGPT: Towards LLMs as Operating Systems — Packer et al. (2023)",
        "url": "https://arxiv.org/abs/2310.08560",
        "content": """
MemGPT introduces a tiered memory system for LLMs analogous to an operating system's
virtual memory management. The system manages three levels:

  1. In-context storage (main context window) — the LLM's immediate working memory,
     limited to the context window size (e.g. 8K tokens for GPT-4-turbo).
  2. External storage — a vector database holding long-term information beyond the
     context window. Retrieved via semantic search when relevant.
  3. In-weights storage — knowledge encoded in model parameters during pre-training.

The core mechanism is a memory controller that decides when to:
  • page-in: retrieve relevant information from external storage into the context
  • page-out: write current context information to external storage
  • summarize: compress conversation history to free context window space

MemGPT uses function calls to explicitly manage memory: RECALL_MEMORY(query),
CORE_MEMORY_APPEND(name, content), CORE_MEMORY_REPLACE(name, old, new),
ARCHIVAL_MEMORY_SEARCH(query), ARCHIVAL_MEMORY_INSERT(content).

Evaluation: MemGPT outperforms baseline GPT-4 in document QA (+23% accuracy),
conversational agents (higher user satisfaction), and long-conversation coherence.
The key insight: treating the LLM as a stateless function and managing ALL state
externally produces dramatically better results over multi-session interactions.
"""
    },
    {
        "id": "generative_agents",
        "source": "Generative Agents: Interactive Simulacra of Human Behavior — Park et al. (2023)",
        "url": "https://arxiv.org/abs/2304.03442",
        "content": """
Generative Agents are computational agents that simulate believable human behavior
by storing, retrieving, and synthesizing memories over time. Each agent maintains
a memory stream — a comprehensive record of all experiences as natural language.

Three memory processes:
  1. Storage:   All perceptions added to memory stream with timestamp + importance score
  2. Retrieval: Top memories retrieved by recency, importance, and relevance
  3. Reflection: Higher-level generalizations synthesized from raw memories

Retrieval scoring function (normalized sum):
  score = α_recency · recency(m) + α_importance · importance(m) + α_relevance · relevance(m,q)
  where recency(m) = 0.99^(hours_elapsed_since_last_retrieval)

Importance scoring: the agent asks the LLM "On a scale of 1-10, how important is
this memory?" and stores the score. High-importance memories (score ≥ 8) trigger
immediate reflection. Reflection synthesizes 3-5 memories into higher-level insights.

The reflection process: every N memories, the agent asks "What are 3 high-level
insights I can infer from the above statements?" These insights are stored back
into the memory stream, creating a hierarchy of abstraction.

Results: agents exhibit believable emergent behaviors — spreading news, planning
events, forming relationships — that arise from the memory architecture rather
than explicit programming. Memory-enabled agents score 78% on social interaction
coherence vs 34% for agents without long-term memory.
"""
    },
    {
        "id": "amem",
        "source": "A-MEM: Agentic Memory for LLM Agents — Xu et al. (2024)",
        "url": "https://arxiv.org/abs/2312.06681",
        "content": """
A-MEM (Agentic Memory) introduces a Zettelkasten-inspired memory system where each
stored memory note is richly annotated with: keywords, contextual descriptions,
and explicit links to related memories. The system builds a graph of interconnected
memory notes rather than a flat vector index.

Memory note structure:
  { content, keywords[], context_description, linked_note_ids[], timestamp,
    importance, embedding, access_history }

Operations:
  • add_note(text): generate keywords + description via LLM, embed, link to similar notes
  • retrieve(query): find semantically similar notes AND their linked neighbors
  • update_note(id, new_content): regenerate metadata, re-embed, update links
  • evolve(note_id): LLM re-evaluates importance based on access history

Graph traversal retrieval: starting from top-k query results, follow links to
connected notes (BFS up to depth 2), giving richer context than flat vector search.

A-MEM outperforms MemGPT on multi-hop reasoning tasks (answer accuracy +15.3%)
because linked notes enable traversal of related memories not directly matched
by the query embedding. On multi-session QA benchmark: A-MEM 71.2% vs baseline 48.9%.
"""
    },
    {
        "id": "hipporag",
        "source": "HippoRAG: Neurologically Inspired Long-Term Memory for RAG — Gutierrez et al. (2024)",
        "url": "https://arxiv.org/abs/2405.14831",
        "content": """
HippoRAG is inspired by the hippocampal-neocortical memory system in the human brain.
It builds a knowledge graph (KG) from documents where:
  • Nodes: named entities extracted by LLM
  • Edges: relationships between co-occurring entities
  • Embeddings: each node encoded with sentence-transformers

Retrieval process (mimics hippocampal indexing):
  1. Personalized PageRank (PPR) initialized with query-matched entity nodes
  2. PPR propagates through the KG, boosting connected entities
  3. Top-ranked passages retrieved from high-PageRank entities

This allows HippoRAG to retrieve passages connected through multi-hop reasoning
(A→B→C) rather than only directly similar passages, mimicking associative memory.

HippoRAG vs standard dense RAG:
  MuSiQue (multi-hop QA): HippoRAG 31.7% vs BM25 12.6% vs DPR 23.4%
  2WikiMultihopQA: 42.1% vs 18.3% vs 27.8%
  HotpotQA:        44.2% vs 37.8% vs 40.1%

The system uses GPT-3.5-turbo for entity extraction and any sentence-transformer
model for node embeddings. The KG structure persists across sessions, functioning
as true long-term memory rather than per-query ephemeral retrieval.
"""
    },
    {
        "id": "memorybank",
        "source": "MemoryBank: Enhancing Large Language Models with Long-Term Memory — Zhong et al. (2023)",
        "url": "https://arxiv.org/abs/2305.10250",
        "content": """
MemoryBank equips LLMs with a long-term memory mechanism inspired by the Ebbinghaus
Forgetting Curve — memories decay over time but are strengthened by repeated retrieval.

Core mechanisms:
  1. Memory storage: all interactions stored as natural language memories with timestamps
  2. Memory retrieval: semantic similarity search + recency weighting
  3. Memory update: retrieved memories are summarized and strengthened
  4. Memory forgetting: memories below strength threshold are pruned

Forgetting curve implementation:
  strength(t) = S_0 · exp(-t / (τ · access_count))
  where S_0 = initial strength, t = time since creation, τ = decay constant,
  access_count = number of times the memory was retrieved and reinforced.

The system maintains a user profile that accumulates personality traits, preferences,
and biographical facts across sessions. Profile is updated by LLM-generated summaries.

Applications: SiliconFriend chatbot with MemoryBank showed:
  - 35% increase in response personalization scores
  - 28% improvement in emotional support quality
  - Users rated long-term coherence 4.2/5 vs 2.8/5 for memoryless baseline
"""
    },
    {
        "id": "rag_lewis",
        "source": "Retrieval-Augmented Generation for Knowledge-Intensive NLP — Lewis et al. (2020)",
        "url": "https://arxiv.org/abs/2005.11401",
        "content": """
RAG (Retrieval-Augmented Generation) combines a parametric seq2seq model (BART-large)
with a non-parametric dense vector retrieval over Wikipedia (21M passages, FAISS index).
The retriever (DPR) and generator (BART) are trained end-to-end.

RAG-Sequence: the same retrieved document is used for the entire output generation.
RAG-Token: a different retrieved document can be used for each output token.

The key advantage for long-term memory RAG: the retrieval index is updatable without
retraining the generator. New memories can be indexed and immediately accessible,
unlike parametric-only models where new knowledge requires expensive fine-tuning.

Standard RAG limitation for LTM: it retrieves from a static corpus without any notion
of per-user memory, temporal ordering, recency weighting, or memory consolidation.
LTM-RAG extends RAG by adding: per-user episodic stores, semantic abstraction layers,
memory decay, and dynamic consolidation from raw episodes to generalized facts.

Results: NQ 44.5 EM, WebQA 45.5, TriviaQA 56.1 — all SOTA at publication time.
"""
    },
    {
        "id": "self_rag",
        "source": "Self-RAG: Learning to Retrieve, Generate and Critique — Asai et al. (2023)",
        "url": "https://arxiv.org/abs/2310.11511",
        "content": """
Self-RAG adds reflection tokens to control when and how retrieval is performed:
  [Retrieve=Yes/No]  — should we retrieve for this span?
  [IsREL=relevant/irrelevant] — is the retrieved passage relevant?
  [IsSUP=supported/partially/not] — is output supported by retrieved passage?
  [IsUSE=5/4/3/2/1] — is the response useful to the user?

For LTM-RAG systems, the Self-RAG paradigm offers a blueprint for memory gating:
instead of always retrieving from all memory tiers, the system learns when each tier
is relevant. Working memory for recency, episodic for personal history, semantic for
preferences, KB for factual grounding.

Adaptive retrieval rate: Self-RAG retrieves for ~20-40% of sentences depending on
task. For LTM-RAG this translates to: don't always query all memory tiers —
query selectively based on the query type and available memory signals.

Results: Self-RAG 7B outperforms ChatGPT on PopQA (54.9 vs 50.8), ASQA (38.2 vs 35.1),
ARC-Challenge (67.3 vs 66.2). Reflection tokens add <5% latency overhead.
"""
    },
    {
        "id": "cognitive_arch",
        "source": "Cognitive Architecture for Language Agents (CoALA) — Sumers et al. (2023)",
        "url": "https://arxiv.org/abs/2309.02427",
        "content": """
CoALA (Cognitive Architecture for Language Agents) provides a unified framework for
language agents inspired by cognitive science, organizing memory into:

  1. Working memory  — current context, goals, intermediate results (in-context)
  2. Episodic memory — records of past actions and experiences (external store)
  3. Semantic memory — general world knowledge and facts (external store)
  4. Procedural memory — skills and action policies (in-weights or code)

Decision cycle: observe → retrieve relevant memories → reason (plan/generate)
→ execute (retrieval/action/learning) → update memory → repeat.

Memory operations in CoALA:
  Internal: chain-of-thought reasoning over working memory
  Retrieval: query episodic/semantic stores for relevant context
  Learning: episodic expansion (new episode), semantic update (new fact),
            procedural update (new action policy)

CoALA maps existing agent systems:
  ReAct → working memory + retrieval + action
  Reflexion → episodic memory + reflection
  Voyager → procedural memory growth
  Generative Agents → all four memory types

Design principle: strong episodic memory enables correction of semantic errors;
strong semantic memory enables efficient retrieval without exhaustive episode search.
"""
    },
    {
        "id": "langchain_mem",
        "source": "LangChain Memory Documentation — LangChain (2024)",
        "url": "https://python.langchain.com/docs/concepts/memory/",
        "content": """
LangChain provides several built-in memory types for conversational agents:

  ConversationBufferMemory     — stores all messages verbatim (grows unbounded)
  ConversationBufferWindowMemory — stores last K turns only (sliding window)
  ConversationSummaryMemory    — LLM-generated progressive summary of conversation
  ConversationSummaryBufferMemory — hybrid: recent verbatim + old summarized
  ConversationTokenBufferMemory — prunes by token count rather than turn count
  VectorStoreRetrieverMemory   — stores messages in vector DB, retrieves by similarity

For LTM-RAG, the closest built-in is VectorStoreRetrieverMemory, but it lacks:
  • User identity isolation (per-user namespacing)
  • Memory decay and reinforcement
  • Episodic → semantic consolidation
  • User profile management
  • Cross-session persistence with temporal metadata

LangChain's LCEL (LangChain Expression Language) chain pattern:
  memory_context = memory.load_memory_variables({"input": query})
  chain = prompt | llm | output_parser
  result = chain.invoke({"context": memory_context, "question": query})
  memory.save_context({"input": query}, {"output": result})
"""
    },
    {
        "id": "chromadb_ltm",
        "source": "ChromaDB: AI-Native Open-Source Vector Database",
        "url": "https://docs.trychroma.com/",
        "content": """
ChromaDB provides the vector storage backbone for LTM-RAG's semantic and episodic tiers.
Key features used:

  Multi-collection: separate collections for episodic_memory, semantic_memory, kb.
  Metadata filtering: WHERE clause to isolate per-user memories.
    e.g. collection.query(where={"user_id": {"$eq": "user_123"}})

  Cosine similarity with HNSW: O(log n) approximate nearest neighbor.
  Persistent storage: SQLite + hnswlib on disk, survives process restarts.

Per-user memory namespacing pattern:
  # Store memory for user
  collection.add(
    documents=["user prefers concise answers"],
    metadatas=[{"user_id": "alice", "type": "preference", "timestamp": 1704067200}],
    ids=["mem_alice_001"]
  )
  # Retrieve only this user's memories
  results = collection.query(
    query_texts=["what does user prefer"],
    where={"user_id": {"$eq": "alice"}},
    n_results=5
  )

For LTM-RAG, ChromaDB stores: semantic facts (long-lived), episodic embeddings
(medium-lived, subject to decay), and KB chunks (permanent, shared across users).
"""
    },
]

# Primary PDF to attempt downloading
PRIMARY_PDF = {
    "url":    "https://arxiv.org/pdf/2310.08560",
    "local":  "./memgpt.pdf",
    "source": "MemGPT: Towards LLMs as Operating Systems — Packer et al. (2023)",
    "url_ref":"https://arxiv.org/abs/2310.08560",
}

# All reference URLs for display
ALL_REFERENCES = [
    ("MemGPT",                          "https://arxiv.org/abs/2310.08560"),
    ("Generative Agents",               "https://arxiv.org/abs/2304.03442"),
    ("A-MEM (Agentic Memory)",          "https://arxiv.org/abs/2312.06681"),
    ("HippoRAG",                        "https://arxiv.org/abs/2405.14831"),
    ("MemoryBank",                      "https://arxiv.org/abs/2305.10250"),
    ("RAG (Lewis et al.)",              "https://arxiv.org/abs/2005.11401"),
    ("Self-RAG",                        "https://arxiv.org/abs/2310.11511"),
    ("CoALA Cognitive Architecture",    "https://arxiv.org/abs/2309.02427"),
    ("LangChain Memory Docs",           "https://python.langchain.com/docs/concepts/memory/"),
    ("ChromaDB Documentation",          "https://docs.trychroma.com/"),
]


In [4]:
"""
═══════════════════════════════════════════════════
Implements all four memory tiers:

  Tier 1 — WorkingMemoryStore    (in-RAM session buffer, no persistence)
  Tier 2 — EpisodicMemoryStore   (SQLite for metadata + ChromaDB for embeddings)
  Tier 3 — SemanticMemoryStore   (ChromaDB, persistent, per-user namespaced)
  Tier 4 — KnowledgeBaseStore    (ChromaDB, persistent, shared across users)

Memory consolidation: EpisodicMemoryStore → SemanticMemoryStore
  Episodes accessed ≥ CONSOLIDATION_THRESHOLD times are distilled into
  semantic facts via LLM summarization.

Memory decay: Episodic records lose relevance over time (Ebbinghaus curve).
  strength(t) = S_0 · 0.5^(age_days / halflife_days)
  Reinforcement: each retrieval boosts strength by REINFORCE_DELTA.
"""

'\n═══════════════════════════════════════════════════\nImplements all four memory tiers:\n\n  Tier 1 — WorkingMemoryStore    (in-RAM session buffer, no persistence)\n  Tier 2 — EpisodicMemoryStore   (SQLite for metadata + ChromaDB for embeddings)\n  Tier 3 — SemanticMemoryStore   (ChromaDB, persistent, per-user namespaced)\n  Tier 4 — KnowledgeBaseStore    (ChromaDB, persistent, shared across users)\n\nMemory consolidation: EpisodicMemoryStore → SemanticMemoryStore\n  Episodes accessed ≥ CONSOLIDATION_THRESHOLD times are distilled into\n  semantic facts via LLM summarization.\n\nMemory decay: Episodic records lose relevance over time (Ebbinghaus curve).\n  strength(t) = S_0 · 0.5^(age_days / halflife_days)\n  Reinforcement: each retrieval boosts strength by REINFORCE_DELTA.\n'

In [14]:
from __future__ import annotations

import os
import json
import math
import time
import uuid
import sqlite3
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


In [15]:

# ══════════════════════════════════════════════════════════════════════
# SHARED EMBEDDING MANAGER
# ══════════════════════════════════════════════════════════════════════

class EmbeddingManager:
    """Singleton-style HuggingFace embedding wrapper shared across all tiers."""

    _instance: Optional["EmbeddingManager"] = None

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._model: Optional[HuggingFaceEmbeddings] = None

    @classmethod
    def get(cls, cfg: Config) -> "EmbeddingManager":
        if cls._instance is None:
            cls._instance = EmbeddingManager(cfg)
        return cls._instance

    def init(self) -> HuggingFaceEmbeddings:
        if self._model is None:
            Log.step("Loading HuggingFace Embeddings (local, zero API cost)")
            Log.info(f"Model : {self.cfg.EMBEDDING_MODEL}")
            Log.info(f"Device: {self.cfg.EMBEDDING_DEVICE}")
            self._model = HuggingFaceEmbeddings(
                model_name=self.cfg.EMBEDDING_MODEL,
                model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
                encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
            )
            Log.ok("Embedding model loaded (~90 MB, cached in ~/.cache/huggingface/)")
        return self._model

    @property
    def model(self) -> HuggingFaceEmbeddings:
        if self._model is None:
            self.init()
        return self._model

    def embed(self, text: str) -> List[float]:
        return self.model.embed_query(text)

    def embed_many(self, texts: List[str]) -> List[List[float]]:
        return self.model.embed_documents(texts)



In [16]:

# ══════════════════════════════════════════════════════════════════════
# TIER 1 — WORKING MEMORY (RAM buffer)
# ══════════════════════════════════════════════════════════════════════

class WorkingMemoryStore:
    """
    In-RAM short-term buffer for the current session.
    Automatically truncates to WORKING_MEMORY_MAX_TURNS when full.
    NOT persisted between sessions.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._sessions: Dict[str, List[WorkingMemoryTurn]] = {}

    def _ensure(self, session_id: str):
        if session_id not in self._sessions:
            self._sessions[session_id] = []

    def add_turn(self, session_id: str, role: str, content: str,
                 metadata: Optional[Dict[str, Any]] = None) -> WorkingMemoryTurn:
        self._ensure(session_id)
        turn = WorkingMemoryTurn(
            turn_id=str(uuid.uuid4())[:8],
            role=role,
            content=content,
            metadata=metadata or {},
        )
        self._sessions[session_id].append(turn)
        # Truncate if over limit
        max_t = self.cfg.WORKING_MEMORY_MAX_TURNS
        if len(self._sessions[session_id]) > max_t * 2:
            self._sessions[session_id] = self._sessions[session_id][-max_t * 2:]
        Log.memory_op(MemoryOperation.STORE, MemoryTier.WORKING,
                      f"[{session_id[:8]}] {role}: {content[:60]}…")
        return turn

    def get_turns(self, session_id: str, last_n: Optional[int] = None) -> List[WorkingMemoryTurn]:
        self._ensure(session_id)
        turns = self._sessions[session_id]
        if last_n is not None:
            turns = turns[-last_n:]
        return turns

    def format_for_prompt(self, session_id: str) -> str:
        turns = self.get_turns(session_id, last_n=self.cfg.WORKING_MEMORY_MAX_TURNS)
        if not turns:
            return "(no current session history)"
        return "\n".join(f"{t.role.upper()}: {t.content}" for t in turns)

    def clear_session(self, session_id: str):
        self._sessions[session_id] = []
        Log.memory_op(MemoryOperation.FORGET, MemoryTier.WORKING,
                      f"session {session_id[:8]} cleared")

    def turn_count(self, session_id: str) -> int:
        return len(self._sessions.get(session_id, []))



In [17]:


# ══════════════════════════════════════════════════════════════════════
# TIER 2 — EPISODIC MEMORY (SQLite metadata + ChromaDB embeddings)
# ══════════════════════════════════════════════════════════════════════

class EpisodicMemoryStore:
    """
    Stores past conversation episodes with temporal metadata in SQLite
    and semantic embeddings in ChromaDB for similarity retrieval.

    Episodic records are subject to:
      - Exponential decay (older = less relevant)
      - Reinforcement (accessed = stronger)
      - Consolidation threshold (frequently accessed → promoted to semantic)
    """

    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg = cfg
        self.em  = em
        self._db_path = cfg.EPISODIC_DB_PATH
        self._chroma_dir = cfg.SEMANTIC_CHROMA_DIR  # shared chroma client
        self._conn: Optional[sqlite3.Connection] = None
        self._chroma_client: Optional[chromadb.PersistentClient] = None
        self._collection: Optional[Any] = None  # chromadb.Collection

    def init(self) -> "EpisodicMemoryStore":
        Path(self.cfg.BASE_DIR).mkdir(parents=True, exist_ok=True)

        # SQLite for metadata
        self._conn = sqlite3.connect(self._db_path, check_same_thread=False)
        self._conn.row_factory = sqlite3.Row
        self._create_tables()

        # ChromaDB for embeddings
        self._chroma_client = chromadb.PersistentClient(
            path=self._chroma_dir,
            settings=Settings(anonymized_telemetry=False),
        )
        self._collection = self._chroma_client.get_or_create_collection(
            name=self.cfg.EPISODIC_COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )
        n = self._collection.count()
        Log.tier(MemoryTier.EPISODIC, f"initialised — {n} episodes in store")
        return self

    def _create_tables(self):
        cur = self._conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS episodes (
                episode_id    TEXT PRIMARY KEY,
                user_id       TEXT NOT NULL,
                session_id    TEXT NOT NULL,
                role          TEXT NOT NULL,
                content       TEXT NOT NULL,
                timestamp     REAL NOT NULL,
                importance    REAL DEFAULT 0.5,
                access_count  INTEGER DEFAULT 0,
                last_accessed REAL,
                consolidated  INTEGER DEFAULT 0,
                metadata_json TEXT DEFAULT '{}'
            )
        """)
        cur.execute("CREATE INDEX IF NOT EXISTS idx_user ON episodes(user_id)")
        cur.execute("CREATE INDEX IF NOT EXISTS idx_session ON episodes(session_id)")
        cur.execute("CREATE INDEX IF NOT EXISTS idx_consolidated ON episodes(consolidated)")
        self._conn.commit()

    def store(self, user_id: str, session_id: str, role: str,
              content: str, importance: float = 0.5,
              metadata: Optional[Dict[str, Any]] = None) -> EpisodicRecord:
        """Store a new episode in SQLite + ChromaDB."""
        ep_id = f"ep_{user_id[:6]}_{str(uuid.uuid4())[:8]}"
        now   = time.time()
        meta  = metadata or {}

        # SQLite
        cur = self._conn.cursor()
        cur.execute("""
            INSERT INTO episodes
            (episode_id, user_id, session_id, role, content, timestamp,
             importance, access_count, last_accessed, consolidated, metadata_json)
            VALUES (?,?,?,?,?,?,?,0,?,0,?)
        """, (ep_id, user_id, session_id, role, content, now,
              importance, now, json.dumps(meta)))
        self._conn.commit()

        # ChromaDB embedding
        try:
            self._collection.add(
                documents=[content],
                ids=[ep_id],
                metadatas=[{
                    "user_id":    user_id,
                    "session_id": session_id,
                    "role":       role,
                    "timestamp":  now,
                    "importance": importance,
                }],
            )
        except Exception as exc:
            Log.warn(f"Episodic Chroma add failed: {exc}")

        rec = EpisodicRecord(
            episode_id=ep_id, user_id=user_id, session_id=session_id,
            role=role, content=content, timestamp=now, importance=importance,
        )
        Log.memory_op(MemoryOperation.STORE, MemoryTier.EPISODIC,
                      f"[{user_id}] {role}: {content[:70]}…")
        return rec

    def retrieve(self, user_id: str, query: str,
                 k: int = 4) -> List[EpisodicRecord]:
        """Retrieve episodic records by semantic similarity + decay scoring."""
        try:
            results = self._collection.query(
                query_texts=[query],
                n_results=min(k * 2, max(1, self._collection.count())),
                where={"user_id": {"$eq": user_id}},
            )
        except Exception:
            return []

        if not results["ids"] or not results["ids"][0]:
            return []

        records: List[EpisodicRecord] = []
        cur = self._conn.cursor()

        for ep_id in results["ids"][0]:
            row = cur.execute(
                "SELECT * FROM episodes WHERE episode_id=?", (ep_id,)
            ).fetchone()
            if row is None:
                continue

            rec = EpisodicRecord(
                episode_id=row["episode_id"],
                user_id=row["user_id"],
                session_id=row["session_id"],
                role=row["role"],
                content=row["content"],
                timestamp=row["timestamp"],
                importance=row["importance"],
                access_count=row["access_count"],
                last_accessed=row["last_accessed"],
                consolidated=bool(row["consolidated"]),
            )

            # Update access count
            cur.execute("""
                UPDATE episodes SET access_count = access_count + 1,
                last_accessed = ? WHERE episode_id = ?
            """, (time.time(), ep_id))
            records.append(rec)

        self._conn.commit()

        # Re-rank by decayed importance score
        records.sort(
            key=lambda r: r.decayed_importance(self.cfg.DECAY_HALFLIFE_DAYS),
            reverse=True,
        )
        Log.memory_op(MemoryOperation.RETRIEVE, MemoryTier.EPISODIC,
                      f"[{user_id}] '{query[:50]}' → {len(records[:k])} episodes")
        return records[:k]

    def get_consolidation_candidates(self, user_id: str) -> List[EpisodicRecord]:
        """Return episodes that meet the consolidation threshold."""
        cur = self._conn.cursor()
        rows = cur.execute("""
            SELECT * FROM episodes
            WHERE user_id=? AND consolidated=0
              AND access_count >= ?
            ORDER BY importance DESC
            LIMIT 20
        """, (user_id, self.cfg.CONSOLIDATION_ACCESS_THRESHOLD)).fetchall()

        return [
            EpisodicRecord(
                episode_id=r["episode_id"], user_id=r["user_id"],
                session_id=r["session_id"], role=r["role"],
                content=r["content"], timestamp=r["timestamp"],
                importance=r["importance"], access_count=r["access_count"],
                consolidated=bool(r["consolidated"]),
            )
            for r in rows
        ]

    def mark_consolidated(self, episode_id: str):
        cur = self._conn.cursor()
        cur.execute(
            "UPDATE episodes SET consolidated=1 WHERE episode_id=?",
            (episode_id,)
        )
        self._conn.commit()

    def prune_old(self, user_id: str):
        """Remove episodic records below decay threshold (memory forgetting)."""
        cur = self._conn.cursor()
        rows = cur.execute(
            "SELECT episode_id, timestamp, importance, access_count FROM episodes "
            "WHERE user_id=? AND consolidated=0",
            (user_id,)
        ).fetchall()

        pruned = 0
        for row in rows:
            rec = EpisodicRecord(
                episode_id=row["episode_id"], user_id=user_id,
                session_id="", role="", content="",
                timestamp=row["timestamp"],
                importance=row["importance"],
                access_count=row["access_count"],
            )
            if rec.decayed_importance(self.cfg.DECAY_HALFLIFE_DAYS) < 0.05:
                cur.execute("DELETE FROM episodes WHERE episode_id=?",
                            (row["episode_id"],))
                try:
                    self._collection.delete(ids=[row["episode_id"]])
                except Exception:
                    pass
                pruned += 1

        if pruned:
            self._conn.commit()
            Log.memory_op(MemoryOperation.FORGET, MemoryTier.EPISODIC,
                          f"[{user_id}] pruned {pruned} decayed episodes")

    def episode_count(self, user_id: str) -> int:
        cur = self._conn.cursor()
        row = cur.execute(
            "SELECT COUNT(*) as c FROM episodes WHERE user_id=?", (user_id,)
        ).fetchone()
        return row["c"] if row else 0



In [18]:

# ══════════════════════════════════════════════════════════════════════
# TIER 3 — SEMANTIC MEMORY (ChromaDB, per-user namespaced)
# ══════════════════════════════════════════════════════════════════════

class SemanticMemoryStore:
    """
    Distilled long-term facts, preferences, and knowledge per user.
    Consolidated from episodic memory or stored directly.
    Facts are reinforced on access and can be updated/removed.

    Fact types: preference | fact | skill | goal | identity
    """

    def __init__(self, cfg: Config, em: EmbeddingManager,
                 chroma_client: chromadb.PersistentClient):
        self.cfg    = cfg
        self.em     = em
        self._client = chroma_client
        self._collection: Optional[Any] = None

    def init(self) -> "SemanticMemoryStore":
        self._collection = self._client.get_or_create_collection(
            name=self.cfg.SEMANTIC_COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )
        n = self._collection.count()
        Log.tier(MemoryTier.SEMANTIC, f"initialised — {n} semantic facts in store")
        return self

    def store(self, user_id: str, content: str, fact_type: str = "fact",
              strength: float = 0.7, confidence: float = 0.8,
              source_episode: Optional[str] = None,
              metadata: Optional[Dict[str, Any]] = None) -> SemanticRecord:
        """Store a semantic fact for a user."""
        fact_id = f"sem_{user_id[:6]}_{str(uuid.uuid4())[:8]}"
        now     = time.time()
        meta    = metadata or {}

        # Check for near-duplicate (cosine > 0.95) before storing
        existing = self._find_near_duplicate(user_id, content, threshold=0.95)
        if existing:
            # Reinforce the existing fact instead of duplicating
            self._reinforce(existing, self.cfg.REINFORCE_DELTA)
            Log.memory_op(MemoryOperation.REINFORCE, MemoryTier.SEMANTIC,
                          f"[{user_id}] reinforced: {content[:60]}")
            return SemanticRecord(
                fact_id=existing, user_id=user_id, content=content,
                fact_type=fact_type, source_episode=source_episode,
                timestamp=now, strength=strength, confidence=confidence,
            )

        self._collection.add(
            documents=[content],
            ids=[fact_id],
            metadatas=[{
                "user_id":        user_id,
                "fact_type":      fact_type,
                "strength":       strength,
                "confidence":     confidence,
                "source_episode": source_episode or "",
                "timestamp":      now,
                "access_count":   0,
                **{k: str(v) for k, v in meta.items()},
            }],
        )
        Log.memory_op(MemoryOperation.STORE, MemoryTier.SEMANTIC,
                      f"[{user_id}][{fact_type}] {content[:70]}…")
        return SemanticRecord(
            fact_id=fact_id, user_id=user_id, content=content,
            fact_type=fact_type, source_episode=source_episode,
            timestamp=now, strength=strength, confidence=confidence,
        )

    def _find_near_duplicate(self, user_id: str, content: str,
                              threshold: float = 0.95) -> Optional[str]:
        """Return fact_id of a near-duplicate if found."""
        try:
            total = self._collection.count()
            if total == 0:
                return None
            res = self._collection.query(
                query_texts=[content],
                n_results=1,
                where={"user_id": {"$eq": user_id}},
            )
            if res["ids"] and res["ids"][0] and res["distances"]:
                dist = res["distances"][0][0]
                # ChromaDB cosine distance: 0 = identical, 1 = orthogonal
                # similarity = 1 - distance
                if (1.0 - dist) >= threshold:
                    return res["ids"][0][0]
        except Exception:
            pass
        return None

    def _reinforce(self, fact_id: str, delta: float):
        """Strengthen a fact on access."""
        try:
            res = self._collection.get(ids=[fact_id], include=["metadatas"])
            if res["metadatas"]:
                meta = dict(res["metadatas"][0])
                meta["strength"] = min(1.0, float(meta.get("strength", 0.5)) + delta)
                meta["access_count"] = int(meta.get("access_count", 0)) + 1
                self._collection.update(ids=[fact_id], metadatas=[meta])
        except Exception as exc:
            Log.warn(f"Reinforce failed: {exc}")

    def retrieve(self, user_id: str, query: str,
                 k: int = 4) -> List[SemanticRecord]:
        """Retrieve semantic facts for a user by similarity."""
        try:
            total = self._collection.count()
            if total == 0:
                return []
            results = self._collection.query(
                query_texts=[query],
                n_results=min(k, total),
                where={"user_id": {"$eq": user_id}},
            )
        except Exception:
            return []

        if not results["ids"] or not results["ids"][0]:
            return []

        records: List[SemanticRecord] = []
        for i, fact_id in enumerate(results["ids"][0]):
            meta = results["metadatas"][0][i] if results.get("metadatas") else {}
            doc  = results["documents"][0][i] if results.get("documents") else ""

            rec = SemanticRecord(
                fact_id=fact_id,
                user_id=user_id,
                content=doc,
                fact_type=meta.get("fact_type", "fact"),
                source_episode=meta.get("source_episode") or None,
                timestamp=float(meta.get("timestamp", 0)),
                strength=float(meta.get("strength", 0.5)),
                access_count=int(meta.get("access_count", 0)),
                confidence=float(meta.get("confidence", 0.8)),
            )
            self._reinforce(fact_id, self.cfg.REINFORCE_DELTA)
            records.append(rec)

        Log.memory_op(MemoryOperation.RETRIEVE, MemoryTier.SEMANTIC,
                      f"[{user_id}] '{query[:50]}' → {len(records)} facts")
        return records

    def update_fact(self, fact_id: str, new_content: str):
        """Update the content of a semantic fact."""
        self._collection.update(
            ids=[fact_id],
            documents=[new_content],
        )
        Log.memory_op(MemoryOperation.UPDATE, MemoryTier.SEMANTIC,
                      f"updated {fact_id}: {new_content[:60]}")

    def delete_fact(self, fact_id: str):
        """Remove a semantic fact."""
        self._collection.delete(ids=[fact_id])
        Log.memory_op(MemoryOperation.FORGET, MemoryTier.SEMANTIC,
                      f"deleted {fact_id}")

    def fact_count(self, user_id: str) -> int:
        try:
            res = self._collection.get(where={"user_id": {"$eq": user_id}})
            return len(res["ids"])
        except Exception:
            return 0



In [19]:


# ══════════════════════════════════════════════════════════════════════
# TIER 4 — KNOWLEDGE BASE (ChromaDB, shared, static)
# ══════════════════════════════════════════════════════════════════════

class KnowledgeBaseStore:
    """
    Permanent, shared-across-users static document corpus.
    Loaded once from CORPUS + optional PDF download.
    Supports MMR retrieval for diversity-aware results.
    """

    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg = cfg
        self.em  = em
        self._store: Optional[Chroma] = None
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg.CHUNK_SIZE,
            chunk_overlap=cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def _try_pdf(self) -> List[Document]:
        try:
            import urllib.request
            from langchain_community.document_loaders import PyPDFLoader
            if not Path(PRIMARY_PDF["local"]).exists():
                Log.info(f"Downloading PDF: {PRIMARY_PDF['url']}")
                urllib.request.urlretrieve(PRIMARY_PDF["url"], PRIMARY_PDF["local"])
                Log.ok(f"PDF saved → {PRIMARY_PDF['local']}")
            pages = PyPDFLoader(PRIMARY_PDF["local"]).load()
            for p in pages:
                p.metadata.update({
                    "source": PRIMARY_PDF["source"],
                    "url":    PRIMARY_PDF["url_ref"],
                })
            Log.ok(f"PDF loaded: {len(pages)} pages")
            return pages
        except Exception as exc:
            Log.warn(f"PDF unavailable ({exc}), using inline corpus")
            return []

    def build(self) -> "KnowledgeBaseStore":
        """Build the knowledge base from corpus + optional PDF."""
        Log.step("Building Knowledge Base (Tier 4)")
        Path(self.cfg.KB_CHROMA_DIR).mkdir(parents=True, exist_ok=True)

        # Collect raw documents
        raw: List[Document] = self._try_pdf()
        for entry in CORPUS:
            raw.append(Document(
                page_content=entry["content"].strip(),
                metadata={"source": entry["source"], "url": entry["url"], "doc_id": entry["id"]},
            ))

        # Chunk + deduplicate
        chunks = self._splitter.split_documents(raw)
        seen: set = set()
        unique: List[Document] = []
        for i, c in enumerate(chunks):
            h = hashlib.md5(c.page_content.encode()).hexdigest()[:10]
            if h not in seen:
                seen.add(h)
                c.metadata["chunk_id"] = i
                unique.append(c)

        # Build ChromaDB store
        client = chromadb.PersistentClient(
            path=self.cfg.KB_CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        try:
            client.delete_collection(self.cfg.KB_COLLECTION)
        except Exception:
            pass

        self._store = Chroma.from_documents(
            documents=unique,
            embedding=self.em.model,
            client=client,
            collection_name=self.cfg.KB_COLLECTION,
            collection_metadata={"hnsw:space": "cosine"},
        )
        n = self._store._collection.count()
        Log.tier(MemoryTier.KB, f"built — {n} chunks indexed")
        return self

    def retrieve(self, query: str, k: int = 4) -> List[Document]:
        """MMR retrieval from knowledge base."""
        retriever = self._store.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": k,
                "fetch_k": self.cfg.FETCH_K,
                "lambda_mult": self.cfg.MMR_LAMBDA,
            },
        )
        docs = retriever.invoke(query)
        Log.memory_op(MemoryOperation.RETRIEVE, MemoryTier.KB,
                      f"'{query[:50]}' → {len(docs)} docs")
        return docs



In [20]:

# ══════════════════════════════════════════════════════════════════════
# USER PROFILE MANAGER
# ══════════════════════════════════════════════════════════════════════

class UserProfileManager:
    """
    Persistent user profile storage (JSON file).
    Tracks preferences, session counts, and known facts.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._path = Path(cfg.USER_PROFILE_PATH)
        self._profiles: Dict[str, Dict[str, Any]] = {}
        self._load()

    def _load(self):
        Path(self.cfg.BASE_DIR).mkdir(parents=True, exist_ok=True)
        if self._path.exists():
            try:
                with open(self._path) as f:
                    self._profiles = json.load(f)
                Log.ok(f"User profiles loaded: {len(self._profiles)} users")
            except Exception:
                self._profiles = {}

    def _save(self):
        with open(self._path, "w") as f:
            json.dump(self._profiles, f, indent=2)

    def get_or_create(self, user_id: str, display_name: str = "") -> UserProfile:
        if user_id not in self._profiles:
            profile = UserProfile(user_id=user_id, display_name=display_name)
            self._profiles[user_id] = self._profile_to_dict(profile)
            self._save()
            Log.memory_op(MemoryOperation.STORE, MemoryTier.SEMANTIC,
                          f"[PROFILE] new user: {user_id}")
        else:
            data = self._profiles[user_id]
            data["last_active"] = time.time()
            self._save()
        return self._dict_to_profile(self._profiles[user_id])

    def update(self, profile: UserProfile):
        self._profiles[profile.user_id] = self._profile_to_dict(profile)
        self._save()

    def start_session(self, user_id: str) -> str:
        """Increment session counter, return new session_id."""
        sid = f"sess_{user_id[:6]}_{str(uuid.uuid4())[:8]}"
        profile = self.get_or_create(user_id)
        profile.total_sessions += 1
        profile.last_active = time.time()
        self.update(profile)
        Log.memory_op(MemoryOperation.UPDATE, MemoryTier.SEMANTIC,
                      f"[PROFILE] {user_id} → session #{profile.total_sessions}")
        return sid

    def update_preference(self, user_id: str, key: str, value: Any):
        """Store or update a user preference."""
        profile = self.get_or_create(user_id)
        profile.preferences[key] = value
        self.update(profile)
        Log.memory_op(MemoryOperation.UPDATE, MemoryTier.SEMANTIC,
                      f"[PROFILE] {user_id} pref: {key}={value}")

    def _profile_to_dict(self, p: UserProfile) -> Dict[str, Any]:
        return {
            "user_id": p.user_id,
            "display_name": p.display_name,
            "created_at": p.created_at,
            "last_active": p.last_active,
            "total_sessions": p.total_sessions,
            "total_turns": p.total_turns,
            "preferences": p.preferences,
            "known_facts": p.known_facts,
            "metadata": p.metadata,
        }

    def _dict_to_profile(self, d: Dict[str, Any]) -> UserProfile:
        return UserProfile(
            user_id=d["user_id"],
            display_name=d.get("display_name", ""),
            created_at=d.get("created_at", time.time()),
            last_active=d.get("last_active", time.time()),
            total_sessions=d.get("total_sessions", 0),
            total_turns=d.get("total_turns", 0),
            preferences=d.get("preferences", {}),
            known_facts=d.get("known_facts", []),
            metadata=d.get("metadata", {}),
        )


In [21]:
"""
═════════════════════════════════════════════════════
Core engine that orchestrates all four memory tiers into a unified
RAG pipeline with true long-term memory management.

Full Pipeline:
  1. Session init     — load/create user profile, start session
  2. Memory retrieval — query all 4 tiers in parallel
  3. Context assembly — fuse retrieved memories into LLM prompt
  4. LLM generation   — generate grounded, memory-aware answer
  5. Memory update    — store new episode, extract semantic facts
  6. Consolidation    — promote frequently-accessed episodes → semantic
  7. Decay pruning    — remove stale low-importance episodes
"""

'\n═════════════════════════════════════════════════════\nCore engine that orchestrates all four memory tiers into a unified\nRAG pipeline with true long-term memory management.\n\nFull Pipeline:\n  1. Session init     — load/create user profile, start session\n  2. Memory retrieval — query all 4 tiers in parallel\n  3. Context assembly — fuse retrieved memories into LLM prompt\n  4. LLM generation   — generate grounded, memory-aware answer\n  5. Memory update    — store new episode, extract semantic facts\n  6. Consolidation    — promote frequently-accessed episodes → semantic\n  7. Decay pruning    — remove stale low-importance episodes\n'

In [22]:

# ══════════════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════════════

IMPORTANCE_SCORING_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a memory importance evaluator. Rate the importance of storing this "
     "interaction in long-term memory on a scale 0.0–1.0.\n"
     "High importance (0.7–1.0): new facts about the user, preferences, significant decisions, "
     "domain knowledge, corrections to prior beliefs.\n"
     "Medium importance (0.4–0.6): general conversation, questions answered, useful context.\n"
     "Low importance (0.0–0.3): greetings, trivial chitchat, repeated information.\n"
     "Respond ONLY with a JSON object: {\"score\": 0.0-1.0, \"reason\": \"brief reason\"}"),
    ("human", "User turn: {user_turn}\nAssistant turn: {assistant_turn}"),
])

SEMANTIC_EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Extract long-term facts and preferences from this conversation turn.\n"
     "For each extracted fact, classify it as one of:\n"
     "  preference — user preference or style choice\n"
     "  fact       — factual statement the user shared\n"
     "  goal       — something the user wants to achieve\n"
     "  identity   — biographical/identity information about the user\n"
     "  skill      — expertise or knowledge domain the user has\n\n"
     "Respond ONLY with valid JSON (no markdown):\n"
     "{\"facts\": [{\"content\": \"...\", \"type\": \"preference|fact|goal|identity|skill\", "
     "\"confidence\": 0.0-1.0}]}\n"
     "Return empty facts list if nothing worth persisting."),
    ("human",
     "User: {user_turn}\nAssistant: {assistant_turn}\n\nExtract persistent facts:"),
])

CONSOLIDATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are summarizing frequently accessed memory episodes into a single semantic fact.\n"
     "Distill the key insight from these episodes into one clear, concise statement.\n"
     "The statement should capture the most important and generalizable information.\n"
     "Respond ONLY with valid JSON: {\"fact\": \"...\", \"type\": \"fact|preference|goal|identity\", "
     "\"confidence\": 0.0-1.0}"),
    ("human",
     "Episodes to consolidate:\n{episodes}\n\nDistill into a single semantic fact:"),
])

MAIN_RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a long-term memory-aware AI assistant. You have access to:\n"
     "  • The user's semantic long-term memories (distilled facts and preferences)\n"
     "  • Relevant past conversation episodes\n"
     "  • A knowledge base of AI/ML research documents\n"
     "  • The current session context\n\n"
     "Rules:\n"
     "1. Personalize your response using the user's preferences and known facts.\n"
     "2. Reference relevant past interactions naturally (e.g. 'As you mentioned before...').\n"
     "3. Ground factual claims in the knowledge base and cite sources as [Source: title].\n"
     "4. If you learn something new about the user, acknowledge it naturally.\n"
     "5. Match the user's preferred communication style if known.\n"
     "6. Be coherent across sessions — refer to past topics when relevant.\n\n"
     "Memory context:\n{memory_context}"),
    ("human", "{question}"),
])

PREFERENCE_EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Detect any user preferences expressed in this message.\n"
     "Examples: verbosity (concise/detailed), topic interests, expertise level, language.\n"
     "Respond ONLY with JSON: {\"preferences\": {\"key\": \"value\", ...}}\n"
     "Return empty dict if no preferences detected."),
    ("human", "User message: {message}"),
])



ValueError: Invalid format specifier in f-string template. Nested replacement fields are not allowed.

In [ ]:

# ══════════════════════════════════════════════════════════════════════
# LTM-RAG ENGINE
# ══════════════════════════════════════════════════════════════════════

class LTMRAGEngine:
    """
    Long-Term Memory RAG Engine.

    Per-query pipeline:
      retrieve_memories() → assemble_context() → generate() →
      score_importance() → store_episode() → extract_semantics() →
      consolidate() → prune_decay()
    """

    def __init__(self, cfg: Config):
        self.cfg     = cfg
        self._parser = StrOutputParser()

        # Component stores
        self.em:      Optional[EmbeddingManager]    = None
        self.working: Optional[WorkingMemoryStore]  = None
        self.episodic:Optional[EpisodicMemoryStore] = None
        self.semantic:Optional[SemanticMemoryStore] = None
        self.kb:      Optional[KnowledgeBaseStore]  = None
        self.profiles:Optional[UserProfileManager]  = None
        self.llm:     Optional[ChatOllama]     = None

        # Active sessions: user_id → session_id
        self._sessions: Dict[str, str] = {}

    # ── Initialization ────────────────────────────────────────────────

    def setup(self) -> "LTMRAGEngine":
        """Full system initialization."""
        Log.banner("LTM-RAG — LONG-TERM MEMORY RAG INITIALIZATION")
        import chromadb
        from chromadb.config import Settings
        from pathlib import Path

        Path(self.cfg.BASE_DIR).mkdir(parents=True, exist_ok=True)

        # Shared embedding manager
        self.em = EmbeddingManager.get(self.cfg)
        self.em.init()

        # Working memory (Tier 1 — RAM only)
        self.working = WorkingMemoryStore(self.cfg)
        Log.tier(MemoryTier.WORKING, "in-RAM session buffer ready")

        # Shared ChromaDB client for episodic + semantic
        Path(self.cfg.SEMANTIC_CHROMA_DIR).mkdir(parents=True, exist_ok=True)
        import chromadb as chromadb_module
        shared_client = chromadb_module.PersistentClient(
            path=self.cfg.SEMANTIC_CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )

        # Episodic memory (Tier 2 — SQLite + ChromaDB)
        self.episodic = EpisodicMemoryStore(self.cfg, self.em)
        self.episodic._chroma_client = shared_client
        self.episodic._collection = shared_client.get_or_create_collection(
            name=self.cfg.EPISODIC_COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )
        self.episodic.init()

        # Semantic memory (Tier 3 — ChromaDB)
        self.semantic = SemanticMemoryStore(self.cfg, self.em, shared_client)
        self.semantic.init()

        # Knowledge base (Tier 4 — ChromaDB, static)
        self.kb = KnowledgeBaseStore(self.cfg, self.em)
        self.kb.build()

        # User profiles
        self.profiles = UserProfileManager(self.cfg)

        # Azure OpenAI LLM
        if self.cfg.is_configured():
            Log.step("Connecting to Azure OpenAI")
            self.llm = ChatOllama(
                base_url=getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
                model=getattr(config, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
                temperature=getattr(config, 'LLM_TEMPERATURE', 0.0),
                num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
            )
            Log.ok(f"LLM ready — deployment: {self.cfg.OLLAMA_MODEL}")
        else:
            Log.warn("Azure OpenAI not configured. Set OLLAMA_BASE_URL + OLLAMA_BASE_URL")

        Log.banner("LTM-RAG SYSTEM READY")
        return self

    def _invoke(self, prompt: ChatPromptTemplate, **kwargs) -> str:
        if self.llm is None:
            return "[LLM not configured]"
        try:
            chain = prompt | self.llm | self._parser
            return chain.invoke(kwargs)
        except Exception as exc:
            Log.err(f"LLM error: {exc}")
            return f"[LLM error: {exc}]"

    def _parse_json(self, raw: str, fallback: Any = None) -> Any:
        try:
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            return json.loads(clean)
        except json.JSONDecodeError:
            return fallback if fallback is not None else {}

    # ── Session Management ────────────────────────────────────────────

    def start_session(self, user_id: str, display_name: str = "") -> str:
        """Create or resume a session for a user."""
        profile = self.profiles.get_or_create(user_id, display_name)
        session_id = self.profiles.start_session(user_id)
        self._sessions[user_id] = session_id
        Log.section(
            f"SESSION START — User: {display_name or user_id} "
            f"| Sessions: {profile.total_sessions} "
            f"| Episodic: {self.episodic.episode_count(user_id)} episodes "
            f"| Semantic: {self.semantic.fact_count(user_id)} facts",
            colour="\033[33m",
        )
        return session_id

    def end_session(self, user_id: str):
        """End a session and run decay pruning."""
        self.episodic.prune_old(user_id)
        self.working.clear_session(self._sessions.get(user_id, ""))
        Log.section(f"SESSION END — User: {user_id}", colour="\033[2m")

    def get_session(self, user_id: str) -> str:
        """Get or create session_id for user."""
        if user_id not in self._sessions:
            self._sessions[user_id] = self.profiles.start_session(user_id)
        return self._sessions[user_id]

    # ── Step 1: Memory Retrieval ───────────────────────────────────────

    def retrieve_memories(self, user_id: str, query: str) -> MemoryRetrievalResult:
        """Query all four memory tiers and return aggregated result."""
        t0 = time.perf_counter()
        session_id = self.get_session(user_id)
        profile    = self.profiles.get_or_create(user_id)

        Log.section(f"MEMORY RETRIEVAL — '{query[:60]}'")

        # Tier 1: Working memory (current session)
        working_turns = self.working.get_turns(
            session_id, last_n=self.cfg.WORKING_MEMORY_MAX_TURNS
        )
        Log.tier(MemoryTier.WORKING, f"{len(working_turns)} turns in session buffer")

        # Tier 2: Episodic memory
        episodic_results = self.episodic.retrieve(
            user_id, query, k=self.cfg.TOP_K_EPISODIC
        )

        # Tier 3: Semantic memory
        semantic_results = self.semantic.retrieve(
            user_id, query, k=self.cfg.TOP_K_SEMANTIC
        )

        # Tier 4: Knowledge base
        kb_results = self.kb.retrieve(query, k=self.cfg.TOP_K_KB)

        elapsed_ms = int((time.perf_counter() - t0) * 1000)

        tiers_used = [MemoryTier.WORKING]
        if episodic_results: tiers_used.append(MemoryTier.EPISODIC)
        if semantic_results:  tiers_used.append(MemoryTier.SEMANTIC)
        if kb_results:        tiers_used.append(MemoryTier.KB)

        return MemoryRetrievalResult(
            query=query,
            user_id=user_id,
            working_context=working_turns,
            episodic_results=episodic_results,
            semantic_results=semantic_results,
            kb_results=kb_results,
            user_profile=profile,
            retrieval_time_ms=elapsed_ms,
            tiers_used=tiers_used,
        )

    # ── Step 2: LLM Generation ────────────────────────────────────────

    def generate_answer(
        self, query: str, memory_result: MemoryRetrievalResult
    ) -> str:
        """Generate a memory-grounded answer using all retrieved context."""
        context = memory_result.format_context(self.cfg)
        answer  = self._invoke(MAIN_RAG_PROMPT, memory_context=context, question=query)
        return answer

    # ── Step 3: Importance Scoring ────────────────────────────────────

    def score_importance(self, user_turn: str, assistant_turn: str) -> Tuple[float, str]:
        """Score how important this interaction is to store."""
        raw = self._invoke(
            IMPORTANCE_SCORING_PROMPT,
            user_turn=user_turn,
            assistant_turn=assistant_turn[:400],
        )
        data = self._parse_json(raw, {"score": 0.5, "reason": "default"})
        score  = float(data.get("score", 0.5))
        reason = data.get("reason", "")
        Log.info(f"Importance score: {score:.2f} — {reason}")
        return score, reason

    # ── Step 4: Store Episode ─────────────────────────────────────────

    def store_interaction(
        self, user_id: str, session_id: str,
        user_turn: str, assistant_turn: str,
        importance: float,
    ):
        """Store both turns of an interaction in episodic memory."""
        # User turn
        self.episodic.store(
            user_id=user_id, session_id=session_id,
            role="user", content=user_turn,
            importance=importance,
        )
        # Assistant turn (slightly lower importance — it's our own response)
        self.episodic.store(
            user_id=user_id, session_id=session_id,
            role="assistant", content=assistant_turn,
            importance=max(0.1, importance - 0.1),
        )
        # Add to working memory
        self.working.add_turn(session_id, "user",      user_turn)
        self.working.add_turn(session_id, "assistant", assistant_turn)

    # ── Step 5: Semantic Extraction ───────────────────────────────────

    def extract_and_store_semantics(
        self, user_id: str, user_turn: str, assistant_turn: str
    ) -> int:
        """Extract long-term facts/preferences and store in semantic memory."""
        raw = self._invoke(
            SEMANTIC_EXTRACTION_PROMPT,
            user_turn=user_turn,
            assistant_turn=assistant_turn[:400],
        )
        data  = self._parse_json(raw, {"facts": []})
        facts = data.get("facts", [])
        count = 0
        for fact in facts:
            content    = fact.get("content", "").strip()
            fact_type  = fact.get("type", "fact")
            confidence = float(fact.get("confidence", 0.8))
            if content and confidence >= self.cfg.MIN_IMPORTANCE_SCORE:
                self.semantic.store(
                    user_id=user_id, content=content,
                    fact_type=fact_type, confidence=confidence,
                )
                count += 1
        if count:
            Log.memory_op(MemoryOperation.STORE, MemoryTier.SEMANTIC,
                          f"Extracted {count} new semantic fact(s)")
        return count

    # ── Step 6: Preference Detection ──────────────────────────────────

    def detect_and_update_preferences(self, user_id: str, message: str):
        """Detect implicit preferences and update user profile."""
        raw  = self._invoke(PREFERENCE_EXTRACTION_PROMPT, message=message)
        data = self._parse_json(raw, {"preferences": {}})
        prefs = data.get("preferences", {})
        for key, value in prefs.items():
            self.profiles.update_preference(user_id, key, value)

    # ── Step 7: Memory Consolidation ─────────────────────────────────

    def consolidate_memories(self, user_id: str) -> int:
        """
        Promote frequently-accessed episodic memories to semantic memory.
        Returns number of consolidations performed.
        """
        candidates = self.episodic.get_consolidation_candidates(user_id)
        if not candidates:
            return 0

        Log.memory_op(
            MemoryOperation.CONSOLIDATE, MemoryTier.EPISODIC,
            f"[{user_id}] {len(candidates)} episodes eligible for consolidation",
        )

        # Group episodes into small batches for distillation
        batch_size = 3
        count = 0
        for i in range(0, min(len(candidates), 9), batch_size):
            batch = candidates[i:i + batch_size]
            episodes_text = "\n".join(
                f"[{ep.role.upper()} — {ep.age_days():.0f}d ago] {ep.content}"
                for ep in batch
            )

            raw  = self._invoke(CONSOLIDATION_PROMPT, episodes=episodes_text)
            data = self._parse_json(raw, {"fact": None})

            if data.get("fact"):
                self.semantic.store(
                    user_id=user_id,
                    content=data["fact"],
                    fact_type=data.get("type", "fact"),
                    strength=0.8,
                    confidence=float(data.get("confidence", 0.75)),
                    source_episode=batch[0].episode_id,
                )
                # Mark episodes as consolidated
                for ep in batch:
                    self.episodic.mark_consolidated(ep.episode_id)
                count += 1
                Log.memory_op(
                    MemoryOperation.CONSOLIDATE, MemoryTier.SEMANTIC,
                    f"Consolidated: {data['fact'][:70]}",
                )

        return count

    # ── Main Query Entry Point ────────────────────────────────────────

    def query(
        self,
        user_id: str,
        question: str,
        run_consolidation: bool = True,
        detect_preferences: bool = True,
    ) -> Dict[str, Any]:
        """
        Full LTM-RAG pipeline for a single query.

        1. Retrieve from all 4 memory tiers
        2. Generate memory-grounded answer
        3. Score importance
        4. Store interaction (episodic + working)
        5. Extract semantic facts
        6. Detect preferences
        7. Consolidate (if enabled)
        8. Prune decayed memories
        """
        t0         = time.time()
        session_id = self.get_session(user_id)

        Log.section(f"LTM-RAG QUERY [{user_id}] — '{question[:60]}'")

        # ── 1. Retrieve ────────────────────────────────────────────────
        memory_result = self.retrieve_memories(user_id, question)

        # ── 2. Generate ────────────────────────────────────────────────
        Log.step("Generating memory-aware answer")
        answer = self.generate_answer(question, memory_result)

        # ── 3. Importance scoring ─────────────────────────────────────
        importance, imp_reason = self.score_importance(question, answer)

        # ── 4. Store interaction ──────────────────────────────────────
        if importance >= self.cfg.MIN_IMPORTANCE_SCORE:
            self.store_interaction(
                user_id, session_id, question, answer, importance
            )
        else:
            # Still add to working memory even if not persisted
            self.working.add_turn(session_id, "user",      question)
            self.working.add_turn(session_id, "assistant", answer)
            Log.info(f"Interaction not persisted (importance {importance:.2f} < threshold)")

        # ── 5. Semantic extraction ────────────────────────────────────
        sem_count = self.extract_and_store_semantics(user_id, question, answer)

        # ── 6. Preference detection ───────────────────────────────────
        if detect_preferences:
            self.detect_and_update_preferences(user_id, question)

        # ── 7. Consolidation ──────────────────────────────────────────
        consolidations = 0
        if run_consolidation:
            consolidations = self.consolidate_memories(user_id)

        # ── 8. Update profile turn count ─────────────────────────────
        profile = self.profiles.get_or_create(user_id)
        profile.update_activity()
        self.profiles.update(profile)

        elapsed = round(time.time() - t0, 2)

        meta = {
            "tiers_used":      memory_result.tiers_used,
            "semantic_count":  len(memory_result.semantic_results),
            "episodic_count":  len(memory_result.episodic_results),
            "kb_count":        len(memory_result.kb_results),
            "working_turns":   len(memory_result.working_context),
            "consolidations":  consolidations,
            "importance":      importance,
            "new_semantics":   sem_count,
            "elapsed":         elapsed,
        }
        Log.answer_box(question, answer, meta)

        return {
            "question":    question,
            "answer":      answer,
            "user_id":     user_id,
            "session_id":  session_id,
            "importance":  importance,
            "memory_meta": meta,
        }

    # ── Memory Inspection Utilities ───────────────────────────────────

    def show_memory_state(self, user_id: str):
        """Print a summary of the user's current memory state."""
        profile   = self.profiles.get_or_create(user_id)
        ep_count  = self.episodic.episode_count(user_id)
        sem_count = self.semantic.fact_count(user_id)
        sess_id   = self.get_session(user_id)
        wm_count  = self.working.turn_count(sess_id)

        Log.section(f"MEMORY STATE — User: {profile.display_name or user_id}")
        print(f"  Sessions total : {profile.total_sessions}")
        print(f"  Turns total    : {profile.total_turns}")
        print(f"  Preferences    : {profile.preferences}")
        Log.tier(MemoryTier.WORKING,  f"{wm_count} turns in current session")
        Log.tier(MemoryTier.EPISODIC, f"{ep_count} episodes persisted")
        Log.tier(MemoryTier.SEMANTIC, f"{sem_count} semantic facts")
        Log.tier(MemoryTier.KB,       "static (shared)")

        # Show top semantic facts
        if sem_count > 0:
            facts = self.semantic.retrieve(user_id, "user", k=5)
            print(f"\n  Top semantic memories:")
            for f in facts:
                print(f"    [{f.fact_type}] {f.content[:80]} (strength={f.strength:.2f})")

    def inject_memory(self, user_id: str, content: str,
                      fact_type: str = "fact", strength: float = 0.9):
        """Manually inject a semantic memory (for testing/bootstrapping)."""
        self.semantic.store(
            user_id=user_id, content=content,
            fact_type=fact_type, strength=strength,
        )
        Log.ok(f"Injected memory for {user_id}: {content[:60]}")

    def forget_user(self, user_id: str):
        """Remove all memories for a user (GDPR-style right to erasure)."""
        # Episodic SQLite
        self.episodic._conn.execute(
            "DELETE FROM episodes WHERE user_id=?", (user_id,)
        )
        self.episodic._conn.commit()
        # Episodic ChromaDB
        try:
            res = self.episodic._collection.get(
                where={"user_id": {"$eq": user_id}}
            )
            if res["ids"]:
                self.episodic._collection.delete(ids=res["ids"])
        except Exception:
            pass
        # Semantic ChromaDB
        try:
            res = self.semantic._collection.get(
                where={"user_id": {"$eq": user_id}}
            )
            if res["ids"]:
                self.semantic._collection.delete(ids=res["ids"])
        except Exception:
            pass
        # Profile
        if user_id in self.profiles._profiles:
            del self.profiles._profiles[user_id]
            self.profiles._save()
        Log.warn(f"All memories deleted for user: {user_id}")


In [ ]:


# ══════════════════════════════════════════════════════════════════════
# DEMO SIMULATION SCENARIOS
# Multi-turn, multi-session, multi-user — showcases all LTM features
# ══════════════════════════════════════════════════════════════════════

ALICE_SESSION_1 = [
    # Alice introduces herself and her interests
    ("alice", "Hi! I'm Alice. I'm a machine learning researcher focused on NLP.",
     "Alice's identity and domain"),
    ("alice", "I prefer detailed, technical answers with formulas when possible.",
     "Alice's preference"),
    ("alice", "What is the MemGPT tiered memory architecture and how does it compare to standard RAG?",
     "Core LTM-RAG knowledge question"),
    ("alice", "I'm currently building a chatbot that needs to remember user preferences across sessions. Any architecture recommendations?",
     "Goal-oriented question, should be stored"),
    ("alice", "What is the retrieval scoring formula in Generative Agents?",
     "Technical question — involves KB retrieval"),
]

ALICE_SESSION_2 = [
    # Alice returns in a new session — system should remember her
    ("alice", "I'm back. Last time we discussed memory architectures for my chatbot project.",
     "Tests episodic recall across sessions"),
    ("alice", "Based on what you know about me, what memory system would best fit my NLP research chatbot?",
     "Personalized answer requiring semantic + episodic memory"),
    ("alice", "How does HippoRAG's PageRank-based retrieval differ from dense vector search?",
     "Technical — KB + semantic memory"),
    ("alice", "I just realized I also need the system to forget outdated information. How does MemoryBank handle forgetting?",
     "Episodic decay concept"),
]

BOB_SESSION_1 = [
    # Bob — different user, different profile
    ("bob", "Hey, I'm Bob. I'm a software engineer just starting to learn about RAG systems.",
     "Bob's identity — beginner"),
    ("bob", "Please keep explanations simple and avoid heavy math.",
     "Bob's preference — contrasts with Alice"),
    ("bob", "What is RAG and why is it useful?",
     "Beginner-level question"),
    ("bob", "How is Long-Term Memory RAG different from regular RAG?",
     "Core concept question"),
    ("bob", "I want to build something that remembers my preferences. Is that hard to implement?",
     "Goal-oriented question"),
]

BOB_SESSION_2 = [
    # Bob returns — tests user isolation (Bob should NOT see Alice's memories)
    ("bob", "I'm back! I've been reading about ChromaDB.",
     "Tests continuity for Bob"),
    ("bob", "Based on what you know about me, how should I get started building a simple memory system?",
     "Personalized to Bob's beginner level"),
    ("bob", "What's the difference between episodic and semantic memory in AI systems?",
     "Concept question tailored to Bob's level"),
]



In [ ]:

# ══════════════════════════════════════════════════════════════════════
# SYSTEM BUILDER
# ══════════════════════════════════════════════════════════════════════

def build_system(cfg: Config) -> LTMRAGEngine:
    engine = LTMRAGEngine(cfg)
    engine.setup()

    # Print reference table
    Log.step("Reference papers / datasets:")
    for i, (title, url) in enumerate(ALL_REFERENCES, 1):
        Log.info(f"  [{i:2d}] {title}")
        Log.info(f"        {url}")

    return engine


# ══════════════════════════════════════════════════════════════════════
# DEMO: FULL MULTI-SESSION SIMULATION
# ══════════════════════════════════════════════════════════════════════

def run_demo(engine: LTMRAGEngine):
    """
    Simulate two users (Alice + Bob) across two sessions each.
    Demonstrates:
      - User profile creation + persistence
      - Episodic memory accumulation
      - Semantic fact extraction
      - Memory consolidation across sessions
      - User isolation (Alice and Bob don't share memories)
      - Personalized responses based on known profile
    """
    # ── ALICE SESSION 1 ───────────────────────────────────────────────
    print(f"\n{C.BOLD}{C.MAG}{'▓'*W}{C.RESET}")
    print(f"{C.BOLD}{C.MAG}  DEMO: ALICE — SESSION 1 (5 turns){C.RESET}")
    print(f"{C.BOLD}{C.MAG}{'▓'*W}{C.RESET}")

    engine.start_session("alice", display_name="Alice (ML Researcher)")
    for user_id, question, note in ALICE_SESSION_1:
        print(f"\n{C.DIM}  [{note}]{C.RESET}")
        _run_query(engine, user_id, question)
        time.sleep(0.3)
    engine.end_session("alice")

    # ── BOB SESSION 1 ─────────────────────────────────────────────────
    print(f"\n{C.BOLD}{C.BLUE}{'▓'*W}{C.RESET}")
    print(f"{C.BOLD}{C.BLUE}  DEMO: BOB — SESSION 1 (5 turns){C.RESET}")
    print(f"{C.BOLD}{C.BLUE}{'▓'*W}{C.RESET}")

    engine.start_session("bob", display_name="Bob (Software Engineer, Beginner)")
    for user_id, question, note in BOB_SESSION_1:
        print(f"\n{C.DIM}  [{note}]{C.RESET}")
        _run_query(engine, user_id, question)
        time.sleep(0.3)
    engine.end_session("bob")

    # Show memory states after session 1
    print(f"\n{C.BOLD}{C.YELLOW}Memory State After Session 1:{C.RESET}")
    engine.show_memory_state("alice")
    engine.show_memory_state("bob")

    # ── ALICE SESSION 2 ───────────────────────────────────────────────
    print(f"\n{C.BOLD}{C.MAG}{'▓'*W}{C.RESET}")
    print(f"{C.BOLD}{C.MAG}  DEMO: ALICE — SESSION 2 (4 turns) — Tests cross-session memory{C.RESET}")
    print(f"{C.BOLD}{C.MAG}{'▓'*W}{C.RESET}")

    engine.start_session("alice", display_name="Alice (ML Researcher)")
    for user_id, question, note in ALICE_SESSION_2:
        print(f"\n{C.DIM}  [{note}]{C.RESET}")
        _run_query(engine, user_id, question)
        time.sleep(0.3)
    engine.end_session("alice")

    # ── BOB SESSION 2 ─────────────────────────────────────────────────
    print(f"\n{C.BOLD}{C.BLUE}{'▓'*W}{C.RESET}")
    print(f"{C.BOLD}{C.BLUE}  DEMO: BOB — SESSION 2 (3 turns) — Tests user isolation{C.RESET}")
    print(f"{C.BOLD}{C.BLUE}{'▓'*W}{C.RESET}")

    engine.start_session("bob", display_name="Bob (Software Engineer, Beginner)")
    for user_id, question, note in BOB_SESSION_2:
        print(f"\n{C.DIM}  [{note}]{C.RESET}")
        _run_query(engine, user_id, question)
        time.sleep(0.3)
    engine.end_session("bob")

    # Final memory states
    print(f"\n{C.BOLD}{C.GREEN}Final Memory State (after all sessions):{C.RESET}")
    engine.show_memory_state("alice")
    engine.show_memory_state("bob")


def _run_query(engine: LTMRAGEngine, user_id: str, question: str):
    try:
        engine.query(
            user_id=user_id,
            question=question,
            run_consolidation=True,
            detect_preferences=True,
        )
    except Exception as exc:
        Log.err(f"Query failed: {exc}")


W = 76


# ══════════════════════════════════════════════════════════════════════
# INTERACTIVE CLI
# ══════════════════════════════════════════════════════════════════════

def run_interactive(engine: LTMRAGEngine, user_id: str = "user"):
    engine.start_session(user_id, display_name=user_id.capitalize())

    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║        LTM-RAG — Long-Term Memory RAG  ·  Interactive CLI       ║")
    print("║  Commands: 'state' | 'inject <fact>' | 'forget' | 'refs' | 'q'  ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    print(C.RESET)
    print(f"{C.DIM}  Memory tiers: Working (RAM) · Episodic (SQLite+Chroma) · "
          f"Semantic (Chroma) · KB (Chroma){C.RESET}\n")

    while True:
        try:
            user_input = input(f"\n{C.BOLD}[{user_id}] You:{C.RESET} ").strip()
        except (EOFError, KeyboardInterrupt):
            engine.end_session(user_id)
            print("\nSession ended. Goodbye!")
            break

        if not user_input:
            continue

        cmd = user_input.lower()

        if cmd in ("quit", "exit", "q"):
            engine.end_session(user_id)
            print("Session ended. Goodbye!")
            break

        if cmd == "state":
            engine.show_memory_state(user_id)
            continue

        if cmd == "refs":
            for i, (t, u) in enumerate(ALL_REFERENCES, 1):
                print(f"  [{i}] {t} — {u}")
            continue

        if cmd == "forget":
            confirm = input("  ⚠  Erase ALL memories for this user? (yes/no): ").strip().lower()
            if confirm == "yes":
                engine.forget_user(user_id)
            continue

        if cmd.startswith("inject "):
            fact = user_input[7:].strip()
            if fact:
                engine.inject_memory(user_id, fact, fact_type="fact")
            continue

        if cmd.startswith("user "):
            user_id = user_input[5:].strip()
            engine.start_session(user_id, display_name=user_id.capitalize())
            print(f"  Switched to user: {user_id}")
            continue

        try:
            engine.query(user_id, user_input,
                         run_consolidation=True, detect_preferences=True)
        except Exception as exc:
            Log.err(f"Error: {exc}")
